# 🧬 Protein Secondary Structure Prediction
## Complete Machine Learning + Deep Learning Project

**Dataset:** Kaggle — Protein Secondary Structure by `alfrandom`  
**Project basis:** Software Requirements Specification (SRS)  
**Primary task:** Three-state (Q3 / `sst3`) residue-level classification

### Structural classes
- **H** — Alpha Helix
- **E** — Beta Strand
- **C** — Coil / Irregular Structure

### SRS-complete workflow
1. Dataset loading and schema verification
2. Data cleaning and validation
3. Exploratory Data Analysis
4. Protein-level 70/15/15 split
5. ML feature engineering
6. Logistic Regression
7. Random Forest
8. Support Vector Machine
9. K-Nearest Neighbors
10. XGBoost
11. ANN
12. 1D CNN
13. BiLSTM
14. Evaluation with Accuracy, Precision, Recall, F1-score and Confusion Matrix
15. ML vs DL comparison
16. Best-model selection
17. Protein-sequence prediction
18. Model persistence
19. Project-ready outputs

## 0. Project Requirements Mapping

| SRS Requirement | Notebook Implementation |
|---|---|
| Dataset | Kaggle Protein Secondary Structure |
| Input | `seq` |
| Target | `sst3` |
| EDA | Length, class, amino-acid distributions |
| Split | 70% train / 15% validation / 15% test |
| ML | Logistic Regression, Random Forest, SVM, KNN, XGBoost |
| DL | ANN, 1D CNN, BiLSTM |
| Metrics | Accuracy, Precision, Recall, F1 |
| Error analysis | Confusion matrices |
| Comparison | ML vs DL table + chart |
| Best model | Automatic selection by weighted F1 |
| Prediction | New protein sequence → H/E/C |
| Persistence | Saved ML/DL models |

In [ ]:
# ============================================================
# 1. IMPORTS, REPRODUCIBILITY AND CONFIGURATION
# ============================================================

import os
import glob
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost is not installed. Installing it in the next cell is recommended.")

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")

print("TensorFlow:", tf.__version__)
print("XGBoost available:", XGB_AVAILABLE)

In [ ]:
# If XGBoost is unavailable, install it in the Kaggle environment.
if not XGB_AVAILABLE:
    !pip -q install xgboost
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True

print("XGBoost ready.")

# 1. Dataset Loading

The Kaggle dataset contains protein sequences and residue-level secondary-structure annotations. The dataset documentation identifies `seq` as the peptide sequence, `sst3` as the three-state structure, `sst8` as the eight-state structure, and includes `pdb_id`, `chain_code`, `len`, and `has_nonstd_aa`. The dataset also provides a PISCES-curated subset of 9,079 sequences intended for training. citeturn0search0turn0search1

This notebook automatically prefers the PISCES file when it is present.

In [ ]:
# ============================================================
# 2. FIND KAGGLE DATASET
# ============================================================

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

print("CSV files found:")
for f in csv_files:
    print(" -", f)

preferred = [
    f for f in csv_files
    if "pdb-intersect-pisces" in os.path.basename(f).lower()
]

if preferred:
    DATA_PATH = preferred[0]
else:
    cleaned = [
        f for f in csv_files
        if "ss.cleaned" in os.path.basename(f).lower()
    ]
    if cleaned:
        DATA_PATH = cleaned[0]
    elif csv_files:
        DATA_PATH = csv_files[0]
    else:
        raise FileNotFoundError(
            "No CSV file found. Attach the Kaggle Protein Secondary Structure dataset."
        )

print("\nSelected:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

# 2. Data Cleaning & Validation

In [ ]:
# ============================================================
# 3. DATA VALIDATION
# ============================================================

required_columns = {"seq", "sst3"}

missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df[["seq", "sst3"]].isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df = df.dropna(subset=["seq", "sst3"]).copy()

df["seq"] = df["seq"].astype(str).str.upper()
df["sst3"] = df["sst3"].astype(str).str.upper()

df["seq_len_actual"] = df["seq"].str.len()
df["sst3_len_actual"] = df["sst3"].str.len()

# Sequence and structural annotation must have equal residue counts.
df = df[
    (df["seq_len_actual"] == df["sst3_len_actual"]) &
    (df["seq_len_actual"] > 0)
].copy()

# Q3 target must contain only C/E/H.
df = df[
    df["sst3"].map(lambda x: set(x).issubset({"C", "E", "H"}))
].copy()

# Protein identity for group-aware splitting.
if {"pdb_id", "chain_code"}.issubset(df.columns):
    df["protein_id"] = (
        df["pdb_id"].astype(str) + "_" +
        df["chain_code"].astype(str)
    )
else:
    df["protein_id"] = np.arange(len(df)).astype(str)

# Remove duplicate protein records.
df = df.drop_duplicates(subset=["protein_id"]).reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("Unique classes:", sorted(set("".join(df["sst3"]))))
print("Length consistency:", (df["seq_len_actual"] == df["sst3_len_actual"]).all())
display(df.head())

# 3. Exploratory Data Analysis

In [ ]:
# ============================================================
# 4. EDA — SEQUENCE LENGTH
# ============================================================

plt.figure(figsize=(10, 5))
sns.histplot(df["seq_len_actual"], bins=50, kde=True)
plt.title("Protein Sequence Length Distribution")
plt.xlabel("Sequence Length")
plt.ylabel("Number of Proteins")
plt.show()

display(df["seq_len_actual"].describe())

In [ ]:
# ============================================================
# 5. EDA — SECONDARY STRUCTURE DISTRIBUTION
# ============================================================

residue_counts = Counter("".join(df["sst3"]))

structure_df = pd.DataFrame(
    list(residue_counts.items()),
    columns=["Structure", "Count"]
).sort_values("Structure")

plt.figure(figsize=(7, 5))
sns.barplot(data=structure_df, x="Structure", y="Count")
plt.title("Residue-Level Secondary Structure Distribution")
plt.xlabel("Structure")
plt.ylabel("Residue Count")
plt.show()

display(structure_df)

In [ ]:
# ============================================================
# 6. EDA — AMINO ACID DISTRIBUTION
# ============================================================

aa_counts = Counter("".join(df["seq"]))

aa_df = pd.DataFrame(
    sorted(aa_counts.items(), key=lambda x: x[1], reverse=True),
    columns=["AminoAcid", "Count"]
)

plt.figure(figsize=(12, 5))
sns.barplot(data=aa_df, x="AminoAcid", y="Count")
plt.title("Amino Acid Frequency")
plt.xlabel("Amino Acid")
plt.ylabel("Count")
plt.show()

display(aa_df)

## 4. Comprehensive EDA Report

This section converts the exploratory analysis into a formal report. It examines:

1. Dataset structure and data quality
2. Missing values and duplicates
3. Protein and sequence statistics
4. Secondary-structure class balance
5. Amino-acid composition
6. Sequence-length patterns
7. Structural composition by protein
8. Relationship between amino acids and secondary structure
9. Rare/unknown residues
10. Key findings and implications for ML/DL

In [ ]:
# ============================================================
# 7. DATASET OVERVIEW REPORT
# ============================================================

eda_report = {
    "Number of protein records": len(df),
    "Number of columns": df.shape[1],
    "Total residues": int(df["seq_len_actual"].sum()),
    "Minimum sequence length": int(df["seq_len_actual"].min()),
    "Maximum sequence length": int(df["seq_len_actual"].max()),
    "Mean sequence length": round(df["seq_len_actual"].mean(), 2),
    "Median sequence length": round(df["seq_len_actual"].median(), 2),
    "Unique protein IDs": df["protein_id"].nunique(),
    "Unique secondary structures": sorted(set("".join(df["sst3"]))),
    "Total duplicate rows after cleaning": int(df.duplicated().sum())
}

report_df = pd.DataFrame(
    list(eda_report.items()),
    columns=["EDA Metric", "Value"]
)

display(report_df)

In [ ]:
# ============================================================
# 8. DATA QUALITY REPORT
# ============================================================

quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": [df[c].dtype for c in df.columns],
    "Missing Values": [df[c].isna().sum() for c in df.columns],
    "Missing %": [round(df[c].isna().mean() * 100, 3) for c in df.columns],
    "Unique Values": [df[c].nunique() for c in df.columns]
})

display(quality_report)

print("Rows after cleaning:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print(
    "Sequence/structure length mismatch:",
    int((df["seq_len_actual"] != df["sst3_len_actual"]).sum())
)
print(
    "Invalid sst3 records:",
    int(
        (~df["sst3"].map(lambda x: set(x).issubset({"C", "E", "H"}))).sum()
    )
)

In [ ]:
# ============================================================
# 9. PROTEIN LENGTH STATISTICAL REPORT
# ============================================================

length_stats = df["seq_len_actual"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).to_frame("Value")

display(length_stats)

skewness = df["seq_len_actual"].skew()
kurtosis = df["seq_len_actual"].kurtosis()

print(f"Sequence-length skewness: {skewness:.4f}")
print(f"Sequence-length kurtosis : {kurtosis:.4f}")

if skewness > 0.5:
    print("Interpretation: Protein lengths are positively/right skewed.")
elif skewness < -0.5:
    print("Interpretation: Protein lengths are negatively/left skewed.")
else:
    print("Interpretation: Protein lengths are approximately symmetric.")

In [ ]:
# ============================================================
# 10. PROTEIN LENGTH DISTRIBUTION — BOXPLOT
# ============================================================

plt.figure(figsize=(12, 3.5))
sns.boxplot(x=df["seq_len_actual"])
plt.title("Protein Sequence Length — Boxplot")
plt.xlabel("Sequence Length")
plt.show()

q1 = df["seq_len_actual"].quantile(0.25)
q3 = df["seq_len_actual"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

length_outliers = df[
    (df["seq_len_actual"] < lower) |
    (df["seq_len_actual"] > upper)
]

print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Potential length outliers: {len(length_outliers)}")

In [ ]:
# ============================================================
# 11. SECONDARY STRUCTURE CLASS BALANCE
# ============================================================

structure_counts = Counter("".join(df["sst3"]))

class_report = pd.DataFrame({
    "Class": ["C", "E", "H"],
    "Residue Count": [
        structure_counts.get("C", 0),
        structure_counts.get("E", 0),
        structure_counts.get("H", 0)
    ]
})

class_report["Percentage"] = (
    class_report["Residue Count"] /
    class_report["Residue Count"].sum() * 100
).round(2)

display(class_report)

imbalance_ratio = (
    class_report["Residue Count"].max() /
    class_report["Residue Count"].min()
)

print(f"Majority/minority class ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 2:
    print("Interpretation: The residue classes show noticeable class imbalance.")
else:
    print("Interpretation: The residue classes are relatively balanced.")

In [ ]:
# ============================================================
# 12. SECONDARY STRUCTURE DISTRIBUTION BY PROTEIN
# ============================================================

protein_structure_stats = []

for row in df.itertuples():
    counts = Counter(row.sst3)
    total = len(row.sst3)

    protein_structure_stats.append({
        "protein_id": row.protein_id,
        "length": total,
        "C_pct": counts.get("C", 0) / total * 100,
        "E_pct": counts.get("E", 0) / total * 100,
        "H_pct": counts.get("H", 0) / total * 100
    })

protein_structure_df = pd.DataFrame(protein_structure_stats)

display(
    protein_structure_df[
        ["length", "C_pct", "E_pct", "H_pct"]
    ].describe().round(2)
)

plt.figure(figsize=(12, 5))
sns.boxplot(
    data=protein_structure_df[
        ["C_pct", "E_pct", "H_pct"]
    ]
)
plt.title("Secondary Structure Composition Across Proteins")
plt.ylabel("Residue Percentage")
plt.show()

In [ ]:
# ============================================================
# 13. AMINO ACID COMPOSITION REPORT
# ============================================================

total_residues = sum(aa_counts.values())

aa_report = aa_df.copy()
aa_report["Percentage"] = (
    aa_report["Count"] / total_residues * 100
).round(3)

display(aa_report)

plt.figure(figsize=(13, 6))
sns.barplot(
    data=aa_report,
    x="AminoAcid",
    y="Percentage"
)
plt.title("Amino Acid Composition (%)")
plt.xlabel("Amino Acid")
plt.ylabel("Percentage of All Residues")
plt.show()

In [ ]:
# ============================================================
# 14. RARE / UNKNOWN AMINO ACIDS REPORT
# ============================================================

standard_aas = set(AMINO_ACIDS)
all_observed_aas = set("".join(df["seq"]))

non_standard_aas = sorted(all_observed_aas - standard_aas)

print("Standard amino acids:", "".join(AMINO_ACIDS))
print("Observed amino acids:", "".join(sorted(all_observed_aas)))
print("Non-standard/unknown symbols:", non_standard_aas)

if non_standard_aas:
    nonstandard_counts = {
        aa: aa_counts[aa]
        for aa in non_standard_aas
    }
    display(
        pd.DataFrame(
            list(nonstandard_counts.items()),
            columns=["Residue", "Count"]
        ).sort_values("Count", ascending=False)
    )
else:
    print("No non-standard amino-acid symbols were observed.")

In [ ]:
# ============================================================
# 15. AMINO ACID × SECONDARY STRUCTURE RELATIONSHIP
# ============================================================

# Count how frequently each amino acid appears in C/E/H.
aa_structure_counts = {
    aa: {"C": 0, "E": 0, "H": 0}
    for aa in sorted(all_observed_aas)
}

for sequence, structure in zip(df["seq"], df["sst3"]):
    for aa, ss in zip(sequence, structure):
        if aa not in aa_structure_counts:
            aa_structure_counts[aa] = {"C": 0, "E": 0, "H": 0}
        aa_structure_counts[aa][ss] += 1

aa_ss_df = pd.DataFrame(aa_structure_counts).T
aa_ss_df = aa_ss_df.reindex(sorted(aa_ss_df.index))

# Convert rows to within-amino-acid percentages.
aa_ss_pct = aa_ss_df.div(
    aa_ss_df.sum(axis=1),
    axis=0
) * 100

plt.figure(figsize=(13, 7))
sns.heatmap(
    aa_ss_pct,
    annot=True,
    fmt=".1f",
    cmap="Blues"
)
plt.title("Secondary Structure Distribution Within Each Amino Acid")
plt.xlabel("Secondary Structure")
plt.ylabel("Amino Acid")
plt.show()

display(aa_ss_pct.round(2))

In [ ]:
# ============================================================
# 16. STRUCTURE RUN / LOCAL PATTERN ANALYSIS
# ============================================================

# A "run" is a consecutive group of the same secondary-structure label.
run_lengths = {"C": [], "E": [], "H": []}

for structure in df["sst3"]:
    if not structure:
        continue

    current = structure[0]
    length = 1

    for char in structure[1:]:
        if char == current:
            length += 1
        else:
            run_lengths[current].append(length)
            current = char
            length = 1

    run_lengths[current].append(length)

run_report = pd.DataFrame({
    label: pd.Series(values)
    for label, values in run_lengths.items()
})

display(run_report.describe().round(2))

run_summary = pd.DataFrame({
    "Structure": list(run_lengths.keys()),
    "Mean Run Length": [
        np.mean(run_lengths[k]) if run_lengths[k] else 0
        for k in run_lengths
    ],
    "Median Run Length": [
        np.median(run_lengths[k]) if run_lengths[k] else 0
        for k in run_lengths
    ],
    "Maximum Run Length": [
        max(run_lengths[k]) if run_lengths[k] else 0
        for k in run_lengths
    ]
}).round(2)

display(run_summary)

In [ ]:
# ============================================================
# 17. CORRELATION / NUMERIC EDA SUMMARY
# ============================================================

numeric_eda = df[
    ["seq_len_actual", "sst3_len_actual"]
].corr()

plt.figure(figsize=(5, 4))
sns.heatmap(
    numeric_eda,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=-1,
    vmax=1
)
plt.title("Numeric Feature Correlation")
plt.show()

print(
    "Sequence length and annotation length correlation:",
    round(numeric_eda.loc["seq_len_actual", "sst3_len_actual"], 4)
)

In [ ]:
# ============================================================
# 18. FORMAL EDA REPORT — KEY FINDINGS
# ============================================================

mean_len = df["seq_len_actual"].mean()
median_len = df["seq_len_actual"].median()
min_len = df["seq_len_actual"].min()
max_len = df["seq_len_actual"].max()

majority_row = class_report.loc[
    class_report["Residue Count"].idxmax()
]

minority_row = class_report.loc[
    class_report["Residue Count"].idxmin()
]

most_common_aa = aa_report.iloc[0]

print("=" * 80)
print("                 PROTEIN SECONDARY STRUCTURE")
print("                     FORMAL EDA REPORT")
print("=" * 80)

print("\n1. DATASET SIZE")
print(f"   • Protein records: {len(df):,}")
print(f"   • Total residues: {total_residues:,}")
print(f"   • Unique protein IDs: {df['protein_id'].nunique():,}")

print("\n2. DATA QUALITY")
print("   • Missing required sequence/target values: 0 after cleaning")
print(f"   • Duplicate rows remaining: {df.duplicated().sum()}")
print(
    "   • Sequence/structure length consistency: "
    f"{(df['seq_len_actual'] == df['sst3_len_actual']).all()}"
)
print("   • Target contains only C/E/H: True")

print("\n3. SEQUENCE LENGTH")
print(f"   • Minimum length: {min_len}")
print(f"   • Maximum length: {max_len}")
print(f"   • Mean length: {mean_len:.2f}")
print(f"   • Median length: {median_len:.2f}")
print(f"   • Skewness: {skewness:.4f}")

print("\n4. SECONDARY STRUCTURE DISTRIBUTION")
print(
    f"   • Majority class: {majority_row['Class']} "
    f"({majority_row['Percentage']:.2f}%)"
)
print(
    f"   • Minority class: {minority_row['Class']} "
    f"({minority_row['Percentage']:.2f}%)"
)
print(f"   • Majority/minority ratio: {imbalance_ratio:.2f}")

print("\n5. AMINO ACID COMPOSITION")
print(
    f"   • Most frequent amino acid: {most_common_aa['AminoAcid']} "
    f"({most_common_aa['Percentage']:.2f}%)"
)
print(f"   • Distinct observed residue symbols: {len(all_observed_aas)}")

print("\n6. STRUCTURAL LOCALITY")
for _, row in run_summary.iterrows():
    print(
        f"   • {row['Structure']}: mean consecutive run "
        f"{row['Mean Run Length']:.2f}, "
        f"maximum run {int(row['Maximum Run Length'])}"
    )

print("\n7. ML/DL IMPLICATIONS")
print("   • Fixed-size local windows are suitable for traditional ML models.")
print("   • CNNs can learn local amino-acid patterns automatically.")
print("   • BiLSTM can model longer-range bidirectional sequence context.")
print("   • Class distribution should be monitored when interpreting metrics.")
print("   • Protein-level splitting reduces information leakage between sets.")

print("\n" + "=" * 80)
print("EDA CONCLUSION")
print("=" * 80)
print(
    "The EDA establishes the quality, distribution, composition, and structural "
    "characteristics of the dataset before model training. These findings support "
    "using both local-window Machine Learning approaches and sequence-based Deep "
    "Learning approaches for three-state protein secondary-structure prediction."
)

# 4. Train / Validation / Test Split

The SRS specifies:

- **70% Training**
- **15% Validation**
- **15% Testing**

The split is performed at the **whole-protein level** rather than randomly splitting individual residues. This avoids placing residues from the same protein into multiple partitions.

In [ ]:
# ============================================================
# 7. 70 / 15 / 15 PROTEIN-LEVEL SPLIT
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)

print(f"Training proteins   : {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Validation proteins : {len(val_df)} ({len(val_df)/len(df):.2%})")
print(f"Testing proteins    : {len(test_df)} ({len(test_df)/len(df):.2%})")

# PART A — MACHINE LEARNING

## 5. ML Feature Engineering

Traditional ML requires fixed-size numeric features.

For every residue, a **9-residue local window** is created:

`4 residues before + target residue + 4 residues after`

Each amino acid is one-hot encoded. This preserves local sequence context while creating a fixed-length vector.

In [ ]:
# ============================================================
# 8. ML WINDOW ENCODING
# ============================================================

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

WINDOW_SIZE = 9
HALF_WINDOW = WINDOW_SIZE // 2

def encode_amino_acid(aa):
    vector = np.zeros(21, dtype=np.float32)

    if aa in AA_TO_INDEX:
        vector[AA_TO_INDEX[aa]] = 1.0
    else:
        vector[-1] = 1.0

    return vector

def make_residue_windows(sequence, labels):
    padding = "X" * HALF_WINDOW
    padded = padding + sequence + padding

    X, y = [], []

    for i, target in enumerate(labels):
        local_window = padded[i:i + WINDOW_SIZE]

        feature = np.concatenate([
            encode_amino_acid(aa)
            for aa in local_window
        ])

        X.append(feature)
        y.append(target)

    return np.asarray(X, dtype=np.float32), np.asarray(y)

def build_ml_dataset(frame, max_residues=None):
    X_parts, y_parts = [], []

    for row in frame.itertuples():
        X_i, y_i = make_residue_windows(row.seq, row.sst3)
        X_parts.append(X_i)
        y_parts.append(y_i)

    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)

    if max_residues is not None and len(y) > max_residues:
        rng = np.random.default_rng(SEED)
        indices = rng.choice(len(y), size=max_residues, replace=False)
        X = X[indices]
        y = y[indices]

    return X, y

# Caps keep CPU-based Kaggle training practical.
X_train_ml, y_train_ml = build_ml_dataset(train_df, max_residues=500_000)
X_val_ml, y_val_ml = build_ml_dataset(val_df, max_residues=150_000)
X_test_ml, y_test_ml = build_ml_dataset(test_df, max_residues=150_000)

print("Train ML:", X_train_ml.shape)
print("Validation ML:", X_val_ml.shape)
print("Test ML:", X_test_ml.shape)

In [ ]:
# ============================================================
# 9. TARGET ENCODING
# ============================================================

ml_encoder = LabelEncoder()
ml_encoder.fit(["C", "E", "H"])

y_train_ml_enc = ml_encoder.transform(y_train_ml)
y_val_ml_enc = ml_encoder.transform(y_val_ml)
y_test_ml_enc = ml_encoder.transform(y_test_ml)

print("Class mapping:")
for label, number in zip(ml_encoder.classes_, range(len(ml_encoder.classes_))):
    print(label, "->", number)

## 6. Machine Learning Models

The SRS explicitly requires:

1. Logistic Regression
2. Random Forest
3. Support Vector Machine
4. K-Nearest Neighbors
5. XGBoost

In [ ]:
# ============================================================
# 10. ML MODEL DEFINITIONS
# ============================================================

ml_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=300,
            solver="saga",
            n_jobs=-1,
            random_state=SEED
        ))
    ]),

    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LinearSVC(
            C=1.0,
            max_iter=3000,
            random_state=SEED
        ))
    ]),

    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=SEED
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softmax",
        num_class=3,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1
    )
}

print("Models:")
for name in ml_models:
    print(" -", name)

In [ ]:
# ============================================================
# 11. TRAIN ALL ML MODELS
# ============================================================

ml_results = []
trained_ml_models = {}

for name, model in ml_models.items():

    print("\n" + "="*60)
    print("Training:", name)
    print("="*60)

    model.fit(X_train_ml, y_train_ml_enc)

    predictions = model.predict(X_test_ml)

    accuracy = accuracy_score(y_test_ml_enc, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test_ml_enc,
        predictions,
        average="weighted",
        zero_division=0
    )

    ml_results.append({
        "Model": name,
        "Type": "Machine Learning",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    trained_ml_models[name] = model

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")

ml_results_df = (
    pd.DataFrame(ml_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(ml_results_df)

In [ ]:
# ============================================================
# 12. ML CLASSIFICATION REPORTS
# ============================================================

for name, model in trained_ml_models.items():

    predictions = model.predict(X_test_ml)

    print("\n" + "="*80)
    print(name)
    print("="*80)

    print(
        classification_report(
            y_test_ml_enc,
            predictions,
            target_names=ml_encoder.classes_,
            digits=4,
            zero_division=0
        )
    )

In [ ]:
# ============================================================
# 13. ML CONFUSION MATRICES
# ============================================================

for name, model in trained_ml_models.items():

    predictions = model.predict(X_test_ml)
    cm = confusion_matrix(y_test_ml_enc, predictions)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=ml_encoder.classes_,
        yticklabels=ml_encoder.classes_
    )

    plt.title(f"{name} — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

# PART B — DEEP LEARNING

The SRS requires three Deep Learning approaches:

- **ANN**
- **1D CNN**
- **BiLSTM**

All three use the same sequence classification objective. Padding is masked so artificial padding does not become a learning signal.

In [ ]:
# ============================================================
# 14. DEEP LEARNING SEQUENCE PREPARATION
# ============================================================

AA_VOCAB = {
    aa: i + 1
    for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")
}

UNKNOWN_ID = len(AA_VOCAB) + 1

TARGET_TO_ID = {
    "C": 0,
    "E": 1,
    "H": 2
}

ID_TO_TARGET = {
    0: "C",
    1: "E",
    2: "H"
}

# Limit sequence length to reduce memory use.
MAX_LENGTH = int(np.percentile(train_df["seq_len_actual"], 95))
MAX_LENGTH = max(64, min(MAX_LENGTH, 512))

print("Maximum sequence length:", MAX_LENGTH)

def encode_sequence(sequence):
    return [
        AA_VOCAB.get(aa, UNKNOWN_ID)
        for aa in sequence
    ]

def encode_structure(structure):
    return [
        TARGET_TO_ID[label]
        for label in structure
    ]

def create_dl_arrays(frame):
    X = np.zeros(
        (len(frame), MAX_LENGTH),
        dtype=np.int32
    )

    y = np.zeros(
        (len(frame), MAX_LENGTH),
        dtype=np.int32
    )

    weights = np.zeros(
        (len(frame), MAX_LENGTH),
        dtype=np.float32
    )

    for i, row in enumerate(frame.itertuples()):

        sequence = row.seq[:MAX_LENGTH]
        structure = row.sst3[:MAX_LENGTH]

        x_ids = encode_sequence(sequence)
        y_ids = encode_structure(structure)

        X[i, :len(x_ids)] = x_ids
        y[i, :len(y_ids)] = y_ids
        weights[i, :len(y_ids)] = 1.0

    return X, y, weights

X_train_dl, y_train_dl, w_train_dl = create_dl_arrays(train_df)
X_val_dl, y_val_dl, w_val_dl = create_dl_arrays(val_df)
X_test_dl, y_test_dl, w_test_dl = create_dl_arrays(test_df)

print("Train:", X_train_dl.shape)
print("Validation:", X_val_dl.shape)
print("Test:", X_test_dl.shape)

## 7. ANN

The ANN provides the fully connected neural-network baseline required by the SRS.

It uses the embedded sequence representation, flattens the sequence, and predicts the residue-level structural labels.

In [ ]:
# ============================================================
# 15. ANN MODEL
# ============================================================

def build_ann():
    model = tf.keras.Sequential([
        layers.Input(shape=(MAX_LENGTH,)),

        layers.Embedding(
            input_dim=UNKNOWN_ID + 1,
            output_dim=32,
            mask_zero=True
        ),

        layers.Flatten(),

        layers.Dense(256, activation="relu"),
        layers.Dropout(0.30),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.20),

        layers.Dense(
            MAX_LENGTH * 3,
            activation="softmax"
        ),

        layers.Reshape((MAX_LENGTH, 3))
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

ann_model = build_ann()
ann_model.summary()

In [ ]:
# ============================================================
# 16. TRAIN ANN
# ============================================================

common_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5
    )
]

ann_history = ann_model.fit(
    X_train_dl,
    y_train_dl,
    sample_weight=w_train_dl,
    validation_data=(
        X_val_dl,
        y_val_dl,
        w_val_dl
    ),
    epochs=15,
    batch_size=64,
    callbacks=common_callbacks,
    verbose=1
)

## 8. 1D CNN

The 1D CNN learns **local sequence motifs** using convolutional filters. This is useful because nearby amino acids can contain patterns associated with secondary structure.

In [ ]:
# ============================================================
# 17. 1D CNN MODEL
# ============================================================

def build_cnn():
    inputs = layers.Input(shape=(MAX_LENGTH,))

    x = layers.Embedding(
        input_dim=UNKNOWN_ID + 1,
        output_dim=64,
        mask_zero=True
    )(inputs)

    # Convert embedding mask-safe sequence into convolution features.
    x = layers.Conv1D(
        filters=128,
        kernel_size=5,
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.Dropout(0.20)(x)

    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.Dropout(0.20)(x)

    outputs = layers.Dense(
        3,
        activation="softmax"
    )(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

cnn_model = build_cnn()
cnn_model.summary()

In [ ]:
# ============================================================
# 18. TRAIN 1D CNN
# ============================================================

cnn_history = cnn_model.fit(
    X_train_dl,
    y_train_dl,
    sample_weight=w_train_dl,
    validation_data=(
        X_val_dl,
        y_val_dl,
        w_val_dl
    ),
    epochs=15,
    batch_size=64,
    callbacks=common_callbacks,
    verbose=1
)

## 9. BiLSTM

The BiLSTM is the sequence-aware model in the project. It reads the protein sequence in both directions, allowing the prediction at a residue to use contextual information from both sides.

In [ ]:
# ============================================================
# 19. BiLSTM MODEL
# ============================================================

def build_bilstm():
    model = tf.keras.Sequential([
        layers.Input(shape=(MAX_LENGTH,)),

        layers.Embedding(
            input_dim=UNKNOWN_ID + 1,
            output_dim=64,
            mask_zero=True
        ),

        layers.Bidirectional(
            layers.LSTM(
                96,
                return_sequences=True,
                dropout=0.20
            )
        ),

        layers.Dropout(0.20),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dropout(0.20),

        layers.Dense(
            3,
            activation="softmax"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

bilstm_model = build_bilstm()
bilstm_model.summary()

In [ ]:
# ============================================================
# 20. TRAIN BiLSTM
# ============================================================

bilstm_history = bilstm_model.fit(
    X_train_dl,
    y_train_dl,
    sample_weight=w_train_dl,
    validation_data=(
        X_val_dl,
        y_val_dl,
        w_val_dl
    ),
    epochs=20,
    batch_size=64,
    callbacks=common_callbacks,
    verbose=1
)

# 10. Deep Learning Training Curves

In [ ]:
# ============================================================
# 21. TRAINING CURVES — ANN
# ============================================================

def plot_history(history, title):
    history_df = pd.DataFrame(history.history)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history_df["loss"], label="Training")
    axes[0].plot(history_df["val_loss"], label="Validation")
    axes[0].set_title(title + " — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(history_df["accuracy"], label="Training")
    axes[1].plot(history_df["val_accuracy"], label="Validation")
    axes[1].set_title(title + " — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(ann_history, "ANN")

In [ ]:
# ============================================================
# 22. TRAINING CURVES — 1D CNN
# ============================================================

plot_history(cnn_history, "1D CNN")

In [ ]:
# ============================================================
# 23. TRAINING CURVES — BiLSTM
# ============================================================

plot_history(bilstm_history, "BiLSTM")

# 11. Deep Learning Evaluation

In [ ]:
# ============================================================
# 24. DL EVALUATION FUNCTION
# ============================================================

def evaluate_dl_model(model, X_test, y_test, weights, model_name):

    probabilities = model.predict(
        X_test,
        batch_size=128,
        verbose=1
    )

    predictions = np.argmax(
        probabilities,
        axis=-1
    )

    mask = weights.astype(bool)

    y_true = y_test[mask]
    y_pred = predictions[mask]

    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print("\n" + "="*70)
    print(model_name)
    print("="*70)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["C", "E", "H"],
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["C", "E", "H"],
        yticklabels=["C", "E", "H"]
    )
    plt.title(model_name + " — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return {
        "Model": model_name,
        "Type": "Deep Learning",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

dl_results = []

dl_results.append(
    evaluate_dl_model(
        ann_model,
        X_test_dl,
        y_test_dl,
        w_test_dl,
        "ANN"
    )
)

dl_results.append(
    evaluate_dl_model(
        cnn_model,
        X_test_dl,
        y_test_dl,
        w_test_dl,
        "1D CNN"
    )
)

dl_results.append(
    evaluate_dl_model(
        bilstm_model,
        X_test_dl,
        y_test_dl,
        w_test_dl,
        "BiLSTM"
    )
)

dl_results_df = (
    pd.DataFrame(dl_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(dl_results_df)

# 12. Final ML vs Deep Learning Comparison

The SRS requires a common comparison of all trained models using:

- Accuracy
- Precision
- Recall
- F1-score

The final model is selected using **weighted F1-score**.

In [ ]:
# ============================================================
# 25. COMBINE ALL RESULTS
# ============================================================

all_results = pd.concat(
    [
        ml_results_df,
        dl_results_df
    ],
    ignore_index=True
)

all_results = (
    all_results
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(all_results)

In [ ]:
# ============================================================
# 26. PERFORMANCE COMPARISON CHART
# ============================================================

comparison_plot = all_results.melt(
    id_vars=["Model", "Type"],
    value_vars=["Accuracy", "Precision", "Recall", "F1"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(14, 6))

sns.barplot(
    data=comparison_plot,
    x="Model",
    y="Score",
    hue="Metric"
)

plt.ylim(0, 1)
plt.title("Machine Learning vs Deep Learning Performance")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=25, ha="right")
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 27. BEST MODEL SELECTION
# ============================================================

best_model_row = all_results.iloc[0]

print("🏆 BEST MODEL")
print("=" * 50)
print("Model    :", best_model_row["Model"])
print("Type     :", best_model_row["Type"])
print("Accuracy :", f"{best_model_row['Accuracy']:.4f}")
print("Precision:", f"{best_model_row['Precision']:.4f}")
print("Recall   :", f"{best_model_row['Recall']:.4f}")
print("F1       :", f"{best_model_row['F1']:.4f}")

# 13. End-to-End Protein Sequence Prediction

In [ ]:
# ============================================================
# 28. DL PREDICTION FUNCTION
# ============================================================

def predict_secondary_structure(sequence, model=bilstm_model):

    sequence = sequence.upper().strip()

    if not sequence:
        raise ValueError("Protein sequence cannot be empty.")

    original_length = len(sequence)

    if len(sequence) > MAX_LENGTH:
        print(
            f"Sequence length {len(sequence)} exceeds MAX_LENGTH={MAX_LENGTH}. "
            f"Truncating for this notebook."
        )
        sequence = sequence[:MAX_LENGTH]

    x = np.zeros(
        (1, MAX_LENGTH),
        dtype=np.int32
    )

    encoded = encode_sequence(sequence)

    x[0, :len(encoded)] = encoded

    probabilities = model.predict(
        x,
        verbose=0
    )[0]

    predicted_ids = np.argmax(
        probabilities[:len(sequence)],
        axis=-1
    )

    predicted_structure = "".join(
        ID_TO_TARGET[i]
        for i in predicted_ids
    )

    return sequence, predicted_structure, original_length

In [ ]:
# ============================================================
# 29. SAMPLE PREDICTION
# ============================================================

sample_sequence = test_df.iloc[0]["seq"]
sample_actual = test_df.iloc[0]["sst3"]

used_sequence, predicted_structure, original_length = (
    predict_secondary_structure(sample_sequence)
)

print("Protein sequence:")
print(used_sequence)

print("\nActual secondary structure:")
print(sample_actual[:len(used_sequence)])

print("\nPredicted secondary structure:")
print(predicted_structure)

sample_accuracy = np.mean(
    np.array(list(sample_actual[:len(used_sequence)])) ==
    np.array(list(predicted_structure))
)

print(f"\nSample residue accuracy: {sample_accuracy:.4f}")

# 14. Model Persistence

The SRS requires a saved trained model. The notebook saves:

- Best traditional ML model
- Label encoder
- ANN
- 1D CNN
- BiLSTM
- Final comparison table

In [ ]:
# ============================================================
# 30. SAVE ALL MODELS AND RESULTS
# ============================================================

import joblib

OUTPUT_DIR = "/kaggle/working/protein_secondary_structure_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save all ML models
for name, trained_model in trained_ml_models.items():

    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    joblib.dump(
        trained_model,
        os.path.join(
            OUTPUT_DIR,
            f"{safe_name}.joblib"
        )
    )

joblib.dump(
    ml_encoder,
    os.path.join(
        OUTPUT_DIR,
        "ml_label_encoder.joblib"
    )
)

# Save DL models
ann_model.save(
    os.path.join(
        OUTPUT_DIR,
        "ann_protein_structure.keras"
    )
)

cnn_model.save(
    os.path.join(
        OUTPUT_DIR,
        "cnn_protein_structure.keras"
    )
)

bilstm_model.save(
    os.path.join(
        OUTPUT_DIR,
        "bilstm_protein_structure.keras"
    )
)

# Save comparison
all_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_model_comparison.csv"
    ),
    index=False
)

print("Saved artifacts:")
for file in sorted(glob.glob(OUTPUT_DIR + "/*")):
    print(" -", file)

# 15. Final SRS Verification Checklist

This cell verifies that every major SRS requirement has been implemented.

| Requirement | Status |
|---|---|
| Kaggle dataset | ✅ |
| Dataset loading | ✅ |
| Data cleaning | ✅ |
| Data validation | ✅ |
| EDA | ✅ |
| Protein-level splitting | ✅ |
| 70/15/15 strategy | ✅ |
| Amino-acid encoding | ✅ |
| Feature engineering | ✅ |
| Logistic Regression | ✅ |
| Random Forest | ✅ |
| SVM | ✅ |
| KNN | ✅ |
| XGBoost | ✅ |
| ANN | ✅ |
| 1D CNN | ✅ |
| BiLSTM | ✅ |
| Accuracy | ✅ |
| Precision | ✅ |
| Recall | ✅ |
| F1-score | ✅ |
| Confusion matrices | ✅ |
| ML comparison | ✅ |
| DL comparison | ✅ |
| ML vs DL comparison | ✅ |
| Best model selection | ✅ |
| Protein prediction | ✅ |
| Model saving | ✅ |
| Future deployment-ready artifacts | ✅ |

In [ ]:
# ============================================================
# 31. FINAL AUTOMATED CHECK
# ============================================================

required_ml = {
    "Logistic Regression",
    "Support Vector Machine",
    "K-Nearest Neighbors",
    "Random Forest",
    "XGBoost"
}

required_dl = {
    "ANN",
    "1D CNN",
    "BiLSTM"
}

trained_ml = set(ml_results_df["Model"])
trained_dl = set(dl_results_df["Model"])

print("ML models completed:", required_ml.issubset(trained_ml))
print("DL models completed:", required_dl.issubset(trained_dl))
print("All required metrics present:",
      {"Accuracy","Precision","Recall","F1"}.issubset(all_results.columns))
print("Final comparison rows:", len(all_results))
print("Best model:", all_results.iloc[0]["Model"])

assert required_ml.issubset(trained_ml)
assert required_dl.issubset(trained_dl)
assert {"Accuracy","Precision","Recall","F1"}.issubset(all_results.columns)

print("\n✅ SRS IMPLEMENTATION CHECK PASSED")

# 16. Conclusion

The completed system implements the SRS from beginning to end.

The project treats protein secondary-structure prediction as a **three-class residue-level classification problem** using the Kaggle `sst3` annotations. The Kaggle dataset documents Q3 as the simplified three-state representation of the DSSP structural states. citeturn0search0

### Machine Learning
Five models are trained:
- Logistic Regression
- Random Forest
- Support Vector Machine
- KNN
- XGBoost

### Deep Learning
Three models are trained:
- ANN
- 1D CNN
- BiLSTM

### Final analysis
All models are evaluated on the isolated test set and compared using Accuracy, Precision, Recall and F1-score. Confusion matrices and training curves provide additional analysis.

### Important academic note
The numerical results are generated by the actual Kaggle execution. They are not hard-coded, so the final report can use the real results produced by the notebook.

# References

1. Kaggle — Protein Secondary Structure, `alfrandom`. citeturn0search0turn0search1
2. Baldi et al. — Bidirectional Dynamics for Protein Secondary Structure Prediction.
3. Chen & Chaudhari — Protein Secondary Structure Prediction with Bidirectional LSTM Networks.
4. RCSB Protein Data Bank / Kabsch-Sander secondary-structure annotations.